# Project 70 — Local Agent Trace Analyzer

**Inspect, debug, and analyze failures in multi-step agent traces — locally.**

| Component | Choice |
|-----------|--------|
| LLM | Ollama (qwen3:8b) |
| Framework | LangGraph |
| Interface | Jupyter |

## 1 · What You Will Learn

1. **Capture and store** agent execution traces
2. **Identify failure patterns** in multi-step workflows
3. Build a **trace analysis framework** for debugging
4. Compute **agent reliability metrics** from traces

## 2 · Why This Project Matters

Multi-step agents are hard to debug. A trace analyzer gives visibility into
the agent's decision process: which step failed, why wrong tool, where hallucination.

## 3 · Setup

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## 4 · Imports

In [ ]:
import json, time
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)
print(f'LLM ready: {MODEL}')

## 5 · Agent Trace Structure

In [ ]:
class AgentTrace:
    def __init__(self, task_id, query):
        self.task_id, self.query = task_id, query
        self.steps, self.t0 = [], time.time()
        self.final_answer = self.success = None
    def add_step(self, step_type, tool=None, inp=None, out=None, error=None):
        self.steps.append({'step': len(self.steps)+1, 'type': step_type, 'tool': tool,
                           'input': str(inp)[:200] if inp else None,
                           'output': str(out)[:200] if out else None,
                           'error': str(error)[:100] if error else None,
                           'time': round(time.time()-self.t0, 3)})
    def finish(self, answer, success): self.final_answer, self.success = answer, success
    def to_dict(self):
        return {'task_id': self.task_id, 'query': self.query, 'steps': self.steps,
                'n_steps': len(self.steps), 'duration': round(time.time()-self.t0,2),
                'answer': self.final_answer, 'success': self.success}
print('AgentTrace ready.')

## 6 · Simulated Agent Executions

In [ ]:
TASKS = [
    {'id': 't1', 'query': 'Tokyo vs NYC population?', 'success': True,
     'steps': [{'type': 'think', 'output': 'Search both populations.'},
               {'type': 'tool_call', 'tool': 'search', 'input': 'Tokyo pop', 'output': '14M city, 37M metro'},
               {'type': 'tool_call', 'tool': 'search', 'input': 'NYC pop', 'output': '8.3M city, 20M metro'},
               {'type': 'answer', 'output': 'Tokyo (14M/37M) > NYC (8.3M/20M)'}]},
    {'id': 't2', 'query': '$300K mortgage at 6.5% 30yr?', 'success': True,
     'steps': [{'type': 'tool_call', 'tool': 'calculator', 'input': 'mortgage(300K,6.5,30)', 'output': '$1,896/mo'},
               {'type': 'answer', 'output': '$1,896.20/month'}]},
    {'id': 't3', 'query': 'Apple earnings summary', 'success': False,
     'steps': [{'type': 'tool_call', 'tool': 'search', 'input': 'Apple Q4', 'error': 'Timeout'},
               {'type': 'tool_call', 'tool': 'search', 'input': 'Apple earnings', 'error': 'Timeout'},
               {'type': 'answer', 'output': 'Unable to retrieve data.'}]},
    {'id': 't4', 'query': 'Translate hello world to French/Spanish', 'success': True,
     'steps': [{'type': 'think', 'output': 'I know this. No tool needed.'},
               {'type': 'tool_call', 'tool': 'search', 'input': 'translate hello', 'output': 'Bonjour/Hola'},
               {'type': 'think', 'output': 'Used search unnecessarily.'},
               {'type': 'answer', 'output': 'French: Bonjour le monde, Spanish: Hola mundo'}]},
]

traces = []
for task in TASKS:
    tr = AgentTrace(task['id'], task['query'])
    for s in task['steps']:
        tr.add_step(s['type'], s.get('tool'), s.get('input'), s.get('output'), s.get('error'))
    tr.finish(task['steps'][-1].get('output',''), task['success'])
    traces.append(tr)
print(f'{len(traces)} traces built')

## 7 · Trace Summary

In [ ]:
trace_data = [t.to_dict() for t in traces]
rows = []
for t in trace_data:
    tc = [s for s in t['steps'] if s['type']=='tool_call']
    errs = [s for s in t['steps'] if s.get('error')]
    rows.append({'task': t['task_id'], 'success': t['success'], 'steps': t['n_steps'],
                 'tools': len(tc), 'errors': len(errs), 'query': t['query'][:35]})
df = pd.DataFrame(rows)
print('TRACE SUMMARY')
print(df.to_string(index=False))

## 8 · Failure Analysis

In [ ]:
failed = [t for t in trace_data if not t['success']]
print(f'Failed: {len(failed)}/{len(trace_data)}')
for t in failed:
    print(f'  {t["task_id"]}: {t["query"]}')
    for s in t['steps']:
        if s.get('error'): print(f'    Step {s["step"]} ERROR: {s["error"]}')

print('\nINEFFICIENCIES:')
for t in trace_data:
    for s in t['steps']:
        if s['type'] == 'think' and 'unnecessarily' in str(s.get('output','')).lower():
            print(f'  {t["task_id"]}: unnecessary tool use detected')

## 9 · LLM-Powered Trace Analysis

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Analyze agent trace. Return JSON: {"efficiency": 1-5, "tool_selection": 1-5, "issues": [...], "suggestions": [...]}'),
    ('human', 'Trace: {trace}'),
])
chain = prompt | llm | StrOutputParser()

for t in trace_data:
    raw = chain.invoke({'trace': json.dumps(t, indent=1)[:1500]})
    try:
        s, e = raw.find('{'), raw.rfind('}') + 1
        analysis = json.loads(raw[s:e]) if s >= 0 else {}
    except: analysis = {'efficiency': 3}
    t['analysis'] = analysis
    print(f'{t["task_id"]}: eff={analysis.get("efficiency","?")}, tools={analysis.get("tool_selection","?")}')

## 10 · Aggregate Metrics

In [ ]:
total_tc = sum(len([s for s in t['steps'] if s['type']=='tool_call']) for t in trace_data)
total_err = sum(len([s for s in t['steps'] if s.get('error')]) for t in trace_data)
print(f'Traces: {len(trace_data)}')
print(f'Success: {sum(1 for t in trace_data if t["success"])}/{len(trace_data)}')
print(f'Tool calls: {total_tc}, Errors: {total_err}')
if total_tc: print(f'Error rate: {total_err/total_tc:.0%}')

## 11 · Save

In [ ]:
with open('agent_traces.json', 'w') as f:
    json.dump(trace_data, f, indent=2)
print('Saved.')

## 12 · Limitations & Key Takeaways

**Limitations:** Simulated traces, no real tool execution, limited error taxonomy

**Takeaways:**
- Agent tracing enables structured debugging of multi-step workflows
- Failure analysis reveals systematic error patterns
- LLM-powered trace analysis automates efficiency review
- Observability is foundational for agent reliability